# 🔗 Session Affinity - Validation Tests

## Overview

Validate **backend-pool session affinity (sticky routing)** for stateful models through the Unified AI API's OpenAI **Responses API** (`/unified-ai/openai/responses`).

When a model is served by a **pool** (2+ backends) and marked `sessionAwareModel: true`, APIM emits a `Set-Cookie: ai-gateway-affinity=...` header on the first response and routes any request that **replays that cookie** back to the **same backend**. This is required for the stateful Responses API, where a follow-up call (referencing a `previous_response_id`) must reach the backend that holds the conversation state.

This notebook:
- **Enables session affinity** for the target model by regenerating the LLM backend onboarding (`sessionAwareModel: true`) into a `.bicepparam` in `validation/.runs/<scenario>/<run-id>/` and redeploying it (only when backends come from `azd`).
- **Provisions an access contract** (APIM product + subscription) scoped to the target model, with `enableResponseHeaders = true` so the responding backend is observable.
- **Establishes 4 sessions × 3 requests** using the **OpenAI SDK** over an **`httpx` client with its own cookie jar** per session — the first request creates a response id, and the 2 follow-ups reuse it (via `previous_response_id`) while the cookie jar keeps them pinned to the same backend.
- **Visualizes** the response id, responding backend, and session cookie for every request.
- **Asserts** that within each session the response id is reused, the same backend answers all 3 requests, and the affinity cookie is maintained.

> **Target model** is selected from explicit `responses` capabilities and a real two-backend pool. Set `target_model_override` to select another eligible deployed model.

> **Prerequisites:** A Citadel Governance Hub with the Unified AI API enabled; the target model onboarded across **2+ backends**; `openai`, `httpx`, and `pandas` installed (`pip install -r ../shared/requirements.txt`). See [LLM Backend Onboarding - Session Affinity Configuration](../bicep/infra/llm-backend-onboarding/README.md#session-affinity-configuration) and [LLM Access Guide - Session affinity](../guides/llm-access-guide.md#backend-pool-types).


### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Annotate deployed model operations with `capabilities`
(or `VALIDATION_MODEL_CAPABILITIES`); unknown capabilities block required coverage.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, and `llm_backends_config` from your active `azd` environment.
- `False` — set explicit resource coordinates in `overrides`.

Set `target_model` to a pooled, session-aware model. Adjust `num_sessions` / `requests_per_session` to change the test matrix.


### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`target_model_override = "my-pooled-responses-model"` optionally pins a model;
`None` selects a Responses-capable model present on at least two compatible backends.
Existing pools need `sessionAwareModel: true`. To configure this test's affinity,
set `run_backend_onboarding = True` and `backend_configuration_complete = True`
after reviewing the azd backend inventory. The generated contract sets stateful
pool settings for the selected model. Example bounds:
`num_sessions = 2; requests_per_session = 3`.

Run All provisions session-aware routing automatically from the complete azd backend inventory (optional source JSON and template options override it). APIM identity is discovered through the SDK. Missing eligible multi-backend Responses coverage is optional: SKIP, with no dependent cloud calls. Skip keywords filter tests only; unrelated and skipped deployments, aliases and global settings are retained.


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from notebookdefaults import backend_source, source_catalog
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    import copy
    source_parameters_file = None  # Optional complete-source JSON override.
    target_model_override = None
    num_sessions = 4
    requests_per_session = 3
    session_cookie_name = "ai-gateway-affinity"
    run_backend_onboarding = True
    backend_configuration_complete = True  # Confirm full effective configuration before shared routing updates.
    contract_business_unit = "Testing"
    contract_use_case_name = "SessionAffinity-" + run_id
    contract_environment = "DEV"
    session_prompts = [
        "My name is Ada and my favorite number is 7. Reply with a short acknowledgement.",
        "What name did I tell you? Answer in one short sentence.",
        "What is my favorite number multiplied by 2? Answer with just the number.",
    ]

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


### 1️⃣ Verify Azure CLI and Connected Subscription


### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'run_backend_onboarding': globals().get('run_backend_onboarding'), 'run_id': globals().get('run_id'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'init_from_azd': globals().get('init_from_azd'), 'target_model_override': globals().get('target_model_override')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    run = ValidationRun("session-affinity")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    utils.require_inputs({f"gateway.{key}": context.config.get(key)
        for key in ("subscription_id", "resource_group", "location", "apim_name")})
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
    source_parameters = backend_source(context, apimClientTool, source_parameters_file,
        overrides.get("backend_template_options"))
    context.catalog = source_catalog(context, source_parameters)
    llm_backends_config = source_parameters["llmBackendConfig"]
    run_dir = repo_root / "validation" / ".runs" / "session-affinity" / run_id
    transport = RequestTransport(timeout=120)
    affinity_ready = False
    target_model = None
    selected_members = set()
    if not 2 <= num_sessions <= 10 or not 3 <= requests_per_session <= 10:
        raise ValueError("Use 2–10 independent sessions and 3–10 requests each.")
    try:
        target_model = context.catalog.select("responses", override=target_model_override, min_backends=2)
        compatible_groups = {}
        for entry in context.catalog.entries:
            if entry.name == target_model and "responses" in entry.capabilities and not context.catalog.is_skipped(entry.name):
                compatible_groups.setdefault((entry.backend_type, entry.protocol), set()).add(entry.backend_id)
        selected_members = next(members for members in compatible_groups.values() if len(members) >= 2)
        affinity_ready = True
    except (ValidationBlocked, StopIteration) as exc:
        if target_model_override:
            raise
        run.skip("model-preflight", "Session affinity needs an eligible Responses model on two compatible backends. " + str(exc))
    utils.format_output("Affinity initialization", {"target_model": target_model,
        "serving_backends": sorted(selected_members), "scenario_ready": affinity_ready,
        "deployment_backends": len(llm_backends_config)}, status="info", part="RESULT")
    run_dir.mkdir(parents=True, exist_ok=True)


### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API, capture the gateway URL, and detect the managed identity used by backend onboarding.


### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`

**Output:** `apimClientTool`, `unified_api`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name` are available to the following cells.


In [ ]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group')}, expected='`apimClientTool`, `unified_api`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name` are available to the following cells.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        unified_api = apimClientTool.discover_api_exact("unified-ai")
        gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
        azure_endpoint = gateway_url + "/"
        apim_resource_gateway_url = gateway_url
        apim_name = apimClientTool.apim_resource_name
        preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
            governance_hub_resource_group, apim_name)}
        owned_products = set()


### 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)

Session affinity only activates when the model is served by a **pool** (2+ backends) **and** marked `sessionAwareModel: true`. This step takes the `llm_backends_config` loaded from `azd`, sets `sessionAwareModel: true` on **every backend that serves `target_model`**, and redeploys the LLM Backend Onboarding Bicep.

The generated parameter file is written with a **`.bicepparam` in `validation/.runs/<scenario>/<run-id>/`** suffix (git-ignored) and includes `configureSessionAffinity = true` + `sessionAffinityDefaults` (cookie `ai-gateway-affinity`). All other backends/models are preserved unchanged. Skipped when `run_backend_onboarding = True` or when no azd backends are available.


### 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)

Generate the editable parameter artifact for 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding) and display the proposed configuration without deploying it.

**Input:** `run_backend_onboarding`, `llm_backends_config`, `selected_members`, `backend_configuration_complete`, `backend`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)', inputs={'run_backend_onboarding': globals().get('run_backend_onboarding'), 'llm_backends_config': globals().get('llm_backends_config'), 'selected_members': globals().get('selected_members'), 'backend_configuration_complete': globals().get('backend_configuration_complete'), 'backend': globals().get('backend')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        onboard_cfg = copy.deepcopy(llm_backends_config)
        serving_backends = []
        for backend in onboard_cfg:
            if backend["backendId"] not in selected_members:
                continue
            models = [m if isinstance(m, dict) else {"name": m} for m in backend.get("supportedModels", [])]
            serving = [m for m in models if m.get("name") == target_model]
            if serving:
                serving_backends.append(backend["backendId"])
                if run_backend_onboarding:
                    for model in serving:
                        model["sessionAwareModel"] = True
                    backend["supportedModels"] = models
                elif not all(m.get("sessionAwareModel") is True for m in serving):
                    run.blocked("affinity-pool", "Enable sessionAwareModel on every serving backend, or opt into complete-config onboarding.")
                    raise ValidationBlocked("Session-aware metadata is missing.")
        if len(set(serving_backends)) < 2:
            run.blocked("affinity-pool", "Responses model must be served by two distinct configured backends.")
            raise ValidationBlocked("Missing eligible multi-backend pool.")
        onboarding_params = None
        if run_backend_onboarding:
            onboarding_parameters = copy.deepcopy(source_parameters)
            onboarding_parameters.update({
                "llmBackendConfig": onboard_cfg,
                "configureSessionAffinity": True,
                "sessionAffinityDefaults": {
                    **source_parameters.get("sessionAffinityDefaults", {}),
                    "cookieName": session_cookie_name, "source": "Cookie",
                },
            })
            live_backends, _ = apimClientTool.get_backends()
            omitted = {item["name"] for item in live_backends} - {item["backendId"] for item in onboard_cfg}
            if omitted:
                raise ValueError(f"Complete source omitted live backends: {sorted(omitted)}")
            onboarding_params = write_parameters(repo_root / "bicep" / "infra" / "llm-backend-onboarding" / "main.bicep",
                onboarding_parameters, run_dir / "affinity-backends")
            utils.format_output('2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)', {'value_1': preview(onboarding_params)}, status='info', part='RESULT')
        else:
            utils.format_output('2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)', {'value_1': "Existing session-aware pool:", 'serving_backends': serving_backends}, status='info', part='RESULT')


### 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding) — continuation 5

Deploy the reviewed artifact for 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding) — continuation 5, then inspect its real resource outputs and ownership.

**Input:** `onboarding_params`, `location`, `run_id`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding) — continuation 5', inputs={'onboarding_params': globals().get('onboarding_params'), 'location': globals().get('location'), 'run_id': globals().get('run_id')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        if onboarding_params:
            deploy_contract(onboarding_params, location, "affinity-backends-" + run_id)
            unified_api = apimClientTool.discover_api_exact("unified-ai")


### 3️⃣ Provision Access Contract

Deploy an access contract (APIM product + subscription) scoped to `target_model`. The product policy uses `set-llm-requested-model` and enforces `allowedModels`, and sets `enableResponseHeaders = true` so responses expose the resolved backend via `UAIG-Backend` (and `x-ms-region`).


### 3️⃣ Provision Access Contract

Generate the editable parameter artifact for 3️⃣ Provision Access Contract and display the proposed configuration without deploying it.

**Input:** `contract_business_unit`, `contract_use_case_name`, `contract_environment`, `run_id`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('3️⃣ Provision Access Contract', inputs={'contract_business_unit': globals().get('contract_business_unit'), 'contract_use_case_name': globals().get('contract_use_case_name'), 'contract_environment': globals().get('contract_environment'), 'run_id': globals().get('run_id')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        affinity_contract = {
            "name": "affinity-" + run_id, "business_unit": contract_business_unit,
            "use_case_name": contract_use_case_name, "environment": contract_environment,
            "use_keyvault": False, "endpoint_secret": "AFFINITY-ENDPOINT", "apikey_secret": "AFFINITY-KEY",
            "description": "Session affinity with observable backend headers",
        }
        product_policy = access_policy([target_model], 20000, 500000, "Monthly",
            inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
        utils.format_output('3️⃣ Provision Access Contract', {'product_policy': product_policy}, status='info', part='RESULT')

        affinity_parameters = access_parameters(context, apimClientTool, affinity_contract, product_policy,
            key_vault=None)

        affinity_params_file = write_parameters(
            repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
            affinity_parameters, run_dir / "affinity")
        utils.format_output('3️⃣ Provision Access Contract', {'value_1': preview(affinity_params_file)}, status='info', part='RESULT')


### 3️⃣ Provision Access Contract — continuation 7

Deploy the reviewed artifact for 3️⃣ Provision Access Contract — continuation 7, then inspect its real resource outputs and ownership.

**Input:** `affinity_params_file`, `location`, `preexisting_products`, `owned_products`, `affinity_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('3️⃣ Provision Access Contract — continuation 7', inputs={'affinity_params_file': globals().get('affinity_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'affinity_contract': globals().get('affinity_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        affinity_outputs = deploy_contract(affinity_params_file, location, affinity_contract["name"],
            required_outputs=("products", "subscriptions"))
        affinity_product_id, affinity_subscription_name = contract_identity(
            affinity_outputs, preexisting_products, owned_products)
        product_id = affinity_product_id


### 4️⃣ Retrieve API Key & Build the OpenAI Base URL

Pull the contract's subscription key and compute the base URL for the OpenAI SDK. The SDK appends `/responses`, so the base URL points at the Unified AI OpenAI-compatible surface: `POST {gateway}/unified-ai/openai/responses`.


### 4️⃣ Retrieve API Key & Build the OpenAI Base URL

Prepare the 4️⃣ Retrieve API Key & Build the OpenAI Base URL state shown below for its subsequent feature-specific requests and assertions.

**Input:** `affinity_subscription_name`, `backend`, `members`, `session_cookie_name`

**Output:** `api_key`, `openai_base_url`, `backend_inventory`, `live_backend_configs`, `eligible_live_pools`, `name` are available to the following cells.


In [ ]:
with utils.notebook_step('4️⃣ Retrieve API Key & Build the OpenAI Base URL', inputs={'affinity_subscription_name': globals().get('affinity_subscription_name'), 'backend': globals().get('backend'), 'members': globals().get('members'), 'session_cookie_name': globals().get('session_cookie_name')}, expected='`api_key`, `openai_base_url`, `backend_inventory`, `live_backend_configs`, `eligible_live_pools`, `name` are available to the following cells.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        api_key = apimClientTool.get_subscription_key(affinity_subscription_name)

        openai_base_url = f"{unified_api['url'].rstrip('/')}/openai"
        utils.format_output('4️⃣ Retrieve API Key & Build the OpenAI Base URL', {'value_1': "Responses SDK base URL:", 'openai_base_url': openai_base_url}, status='info', part='RESULT')
        # Confirm actual pool membership and native affinity configuration, not metadata alone.
        # The native sessionAffinity shape requires the same preview version used by Bicep.
        backend_inventory = utils.run(
            f'az rest --method get --uri "{apimClientTool.apim_service_id}/backends?api-version=2025-03-01-preview" -o json',
            print_output=False)
        if not backend_inventory.success or not isinstance(backend_inventory.json_data, dict):
            raise ValidationBlocked("Unable to inspect live affinity pools.")
        live_backend_configs = {backend["name"]: backend["properties"]
                                for backend in backend_inventory.json_data.get("value", [])}
        eligible_live_pools = {}
        for name, backend in live_backend_configs.items():
            pool = backend.get("pool") or {}
            members = {service["id"].rsplit("/", 1)[-1] for service in pool.get("services", [])}
            affinity = pool.get("session_affinity", pool.get("sessionAffinity")) or {}
            session_id = affinity.get("session_id", affinity.get("sessionId")) or {}
            if len(members) >= 2 and set(serving_backends) <= members and session_id.get("name") == session_cookie_name:
                eligible_live_pools[name] = members
        if not eligible_live_pools:
            run.blocked("live-affinity-pool", "No live multi-member pool has the configured native affinity cookie.")
            raise ValidationBlocked("Configured metadata does not prove live session affinity.")


### 5️⃣ Stateful Session Client & Test Harness

Each **session** gets its own `httpx.Client` (a private cookie jar) wrapped by an `OpenAI` client. Because `httpx` stores `Set-Cookie` and replays it automatically, every request in a session carries the `ai-gateway-affinity` cookie set by the first response — pinning the whole session to one backend.

For each request we use `client.responses.with_raw_response.create(...)` so we can read both the parsed response (its `id`) **and** the raw headers (`UAIG-Backend`, `x-ms-region`, `Set-Cookie`). The first request creates the anchor response id; the follow-ups reuse it via `previous_response_id`, so they only succeed if they reach the backend that stored it.


### 5️⃣ Stateful Session Client & Test Harness

Prepare the 5️⃣ Stateful Session Client & Test Harness state shown below for its subsequent feature-specific requests and assertions.

**Input:** `pool`, `requests_per_session`, `openai_base_url`, `session_prompts`, `target_model`, `session_cookie_name`

**Output:** `make_session_client`, `run_session` are available to the following cells.


In [ ]:
with utils.notebook_step('5️⃣ Stateful Session Client & Test Harness', inputs={'pool': globals().get('pool'), 'requests_per_session': globals().get('requests_per_session'), 'openai_base_url': globals().get('openai_base_url'), 'session_prompts': globals().get('session_prompts'), 'target_model': globals().get('target_model'), 'session_cookie_name': globals().get('session_cookie_name')}, expected='`make_session_client`, `run_session` are available to the following cells.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        import httpx
        from openai import OpenAI


        def _backend_fingerprint(headers):
            """Identify which backend answered. UAIG-Backend is the resolved pool id; x-ms-region
            distinguishes pool members that live in different regions (the common HA layout)."""
            pool   = headers.get('UAIG-Backend', 'n/a')
            region = headers.get('x-ms-region', 'n/a')
            return pool, region


        def make_session_client():
            """A fresh OpenAI client backed by its own httpx cookie jar = one sticky session."""
            http_client = httpx.Client(timeout=120.0)   # persists Set-Cookie across calls in this session
            client = OpenAI(
                api_key="apim-uses-subscription-key",   # placeholder; APIM authenticates via the api-key header
                base_url=openai_base_url,
                http_client=http_client,
                default_headers={"api-key": api_key},
                max_retries=0,                            # no SDK retries — keep routing observations clean
            )
            return client, http_client


        def run_session(session_idx):
            """Run one sticky session of `requests_per_session` Responses API calls and return per-request records."""
            client, http_client = make_session_client()
            records = []
            anchor_id = None
            for req_idx in range(requests_per_session):
                prompt = session_prompts[req_idx % len(session_prompts)]
                kwargs = {"model": target_model, "input": prompt, "store": True}
                if req_idx > 0 and anchor_id:
                    kwargs["previous_response_id"] = anchor_id   # follow-ups reuse the FIRST response id
                prev_used = kwargs.get("previous_response_id", "")

                status, resp_id, pool, region, err = None, "", "n/a", "n/a", ""
                parsed = None
                try:
                    raw = client.responses.with_raw_response.create(**kwargs)
                    status = getattr(raw, "status_code", None) or raw.http_response.status_code
                    pool, region = _backend_fingerprint(raw.headers)
                    parsed = raw.parse()
                    resp_id = getattr(parsed, "id", "") or ""
                    if req_idx == 0:
                        anchor_id = resp_id
                except Exception as e:
                    err = str(e)[:300]
                    resp = getattr(e, "response", None)
                    if resp is not None:
                        status = getattr(resp, "status_code", None)
                        try:
                            pool, region = _backend_fingerprint(resp.headers)
                        except Exception:
                            pass
                    utils.print_error(f"[session {session_idx + 1} req {req_idx + 1}] {err}")

                cookie = http_client.cookies.get(session_cookie_name) or ""
                records.append({
                    "session": session_idx + 1,
                    "request": req_idx + 1,
                    "status": status,
                    "response_id": resp_id,
                    "previous_response_id": prev_used,
                    "anchor_response_id": anchor_id or "",
                    "backend_pool": pool,
                    "backend_region": region,
                    "cookie": cookie,
                    "output_text": getattr(parsed, "output_text", "") if not err else "",
                    "error": err,
                })
                short = (resp_id[:22] + "…") if len(resp_id) > 23 else resp_id
                utils.print_info(f"  session {session_idx + 1} · req {req_idx + 1} → {status} | "
                                 f"id={short or 'n/a'} | region={region} | cookie={'set' if cookie else 'none'}")
            client.close()
            http_client.close()
            return records


### 6️⃣ Run the Sessions (4 sessions × 3 requests)

Each session is independent (its own cookie jar). Watch the `region` and `cookie` stay constant within a session.


### 6️⃣ Run the Sessions (4 sessions × 3 requests)

Prepare the 6️⃣ Run the Sessions (4 sessions × 3 requests) state shown below for its subsequent feature-specific requests and assertions.

**Input:** `num_sessions`, `requests_per_session`

**Output:** `all_records`, `s` are available to the following cells.


In [ ]:
with utils.notebook_step('6️⃣ Run the Sessions (4 sessions × 3 requests)', inputs={'num_sessions': globals().get('num_sessions'), 'requests_per_session': globals().get('requests_per_session')}, expected='`all_records`, `s` are available to the following cells.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        all_records = []
        for s in range(num_sessions):
            utils.print_info(f"▶ Session {s + 1}/{num_sessions}")
            all_records.extend(run_session(s))

        utils.print_ok(f"Completed {num_sessions} sessions × {requests_per_session} requests = {len(all_records)} requests.")


### 7️⃣ Visualize — Response ID, Backend & Cookie per Request

Every row is one request. `Region`/`Backend Pool` identify the responding backend; `Affinity Cookie` is the value stored in the session's cookie jar.


### 7️⃣ Visualize — Response ID, Backend & Cookie per Request

Prepare the 7️⃣ Visualize — Response ID, Backend & Cookie per Request state shown below for its subsequent feature-specific requests and assertions.

**Input:** `all_records`, `target_model`, `num_sessions`, `requests_per_session`

**Output:** `rows`, `df`, `hdr`, `row` are available to the following cells.


In [ ]:
with utils.notebook_step('7️⃣ Visualize — Response ID, Backend & Cookie per Request', inputs={'all_records': globals().get('all_records'), 'target_model': globals().get('target_model'), 'num_sessions': globals().get('num_sessions'), 'requests_per_session': globals().get('requests_per_session')}, expected='`rows`, `df`, `hdr`, `row` are available to the following cells.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        def _short(v, n=18):
            v = str(v or "")
            return v if len(v) <= n else v[:n] + "…"

        rows = [{
            "Session": r["session"],
            "Req": r["request"],
            "Status": r["status"],
            "Response ID": _short(r["response_id"], 26),
            "Prev Resp ID": _short(r["previous_response_id"], 26),
            "Backend Pool": _short(r["backend_pool"], 24),
            "Region": r["backend_region"],
            "Affinity Cookie": _short(r["cookie"], 20),
        } for r in all_records]

        try:
            import pandas as pd
            from IPython.display import display
            df = pd.DataFrame(rows)
            display(df.style.hide(axis="index").set_caption(
                f"Session affinity — {target_model} ({num_sessions} sessions × {requests_per_session} requests)"))
        except Exception:
            hdr = list(rows[0].keys()) if rows else []
            utils.format_output('7️⃣ Visualize — Response ID, Backend & Cookie per Request', {'value_1': " | ".join(hdr)}, status='info', part='RESULT')
            utils.format_output('7️⃣ Visualize — Response ID, Backend & Cookie per Request', {'value_1': "-" * 100}, status='info', part='RESULT')
            for row in rows:
                utils.format_output('7️⃣ Visualize — Response ID, Backend & Cookie per Request', {'value_1': " | ".join(str(row[h]) for h in hdr)}, status='info', part='RESULT')


### 8️⃣ Assertions — Affinity Held

Require successful, nonempty Responses calls, preserved `previous_response_id`,
conversation recall, and a stable **opaque native affinity cookie** from a live,
eligible pool. Inspect its actual configured members and `sessionAffinity`.
`UAIG-Backend` identifies the pool; equal regions do not prove member identity.


### 8️⃣ Assertions — Affinity Held

Exercise 8️⃣ Assertions — Affinity Held and compare the actual response/evidence with the expected feature behavior.

**Input:** `all_records`, `num_sessions`, `r`, `requests_per_session`, `eligible_live_pools`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8️⃣ Assertions — Affinity Held', inputs={'all_records': globals().get('all_records'), 'num_sessions': globals().get('num_sessions'), 'r': globals().get('r'), 'requests_per_session': globals().get('requests_per_session'), 'eligible_live_pools': globals().get('eligible_live_pools')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        from collections import defaultdict
        by_session = defaultdict(list)
        for record in all_records:
            by_session[record["session"]].append(record)
        run.check("session-coverage", len(by_session) == num_sessions,
            expected=num_sessions, actual=len(by_session))
        for sid in range(1, num_sessions + 1):
            recs = sorted(by_session[sid], key=lambda r: r["request"])
            if not recs:
                run.blocked(f"session/{sid}", "No session requests executed.")
                continue
            anchor = recs[0]["response_id"]
            cookie_values = [r["cookie"] for r in recs]
            stable_cookie = bool(cookie_values and cookie_values[0]) and len(set(cookie_values)) == 1
            complete = len(recs) == requests_per_session and all(
                r["status"] == 200 and r["response_id"] and r["output_text"] for r in recs)
            chained = bool(anchor) and all(r["previous_response_id"] == anchor for r in recs[1:])
            recalled = "ada" in recs[1]["output_text"].lower() and "14" in recs[2]["output_text"]
            # UAIG-Backend reports the pool; the native affinity cookie must not be decoded/guessed.
            pools = {r["backend_pool"] for r in recs}
            pool_known = len(pools) == 1 and next(iter(pools)) in eligible_live_pools
            run.check(f"session/{sid}", complete and chained and recalled and stable_cookie and pool_known,
                expected="Complete response chaining; remembered Ada/14; stable native cookie; live eligible pool",
                actual=[r["status"] for r in recs], model=target_model, observed_pools=sorted(pools),
                eligible_members=serving_backends, chained=chained, state_recalled=recalled, stable_cookie=stable_cookie)
        run.skip("individual-backend-identity", "Gateway reports pool identity and an opaque native cookie, not individual member identity. Equal region is not member proof.")


### 9️⃣ Summary Table

One row per session: the anchor **response id**, the **backend** that served it, and the **session cookie**.


### 9️⃣ Summary Table

Collect the final 9️⃣ Summary Table verdict without discarding earlier failed or blocked required cases.

**Input:** `recs`, `by_session`, `sid`, `r`, `s`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('9️⃣ Summary Table', inputs={'recs': globals().get('recs'), 'by_session': globals().get('by_session'), 'sid': globals().get('sid'), 'r': globals().get('r'), 's': globals().get('s')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        summary = []
        for sid, recs in sorted(by_session.items()):
            recs = sorted(recs, key=lambda x: x["request"])
            first = recs[0]
            regions = sorted({r["backend_region"] for r in recs})
            cookies = sorted({r["cookie"] for r in recs if r["cookie"]})
            cookie_disp = "none"
            if cookies:
                cookie_disp = cookies[0] if len(cookies[0]) <= 30 else cookies[0][:29] + "…"
            summary.append({
                "Session": sid,
                "Response ID (anchor)": first["response_id"] or "n/a",
                "Backend Region": regions[0] if len(regions) == 1 else ",".join(regions),
                "Affinity Cookie": cookie_disp,
                "Requests": len(recs),
                "All 200": all(r["status"] == 200 for r in recs),
            })

        try:
            import pandas as pd
            from IPython.display import display
            display(pd.DataFrame(summary).style.hide(axis="index").set_caption("Session Affinity Summary"))
        except Exception:
            for s in summary:
                utils.format_output('9️⃣ Summary Table', {'s': s}, status='info', part='RESULT')

        run.finish()


### 🔟 Cleanup (disabled by default)

Deletes **only** the access contract product and its subscription. The **backend onboarding is left intact** (the target model stays session-aware). Set `cleanup_delete_contract = True` in the init cell to enable.


### 🔟 Cleanup (disabled by default)

Apply the explicitly enabled 🔟 Cleanup (disabled by default) action only to resources owned by this run.

**Input:** `owned_products`, `cleanup_delete_contract`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('🔟 Cleanup (disabled by default)', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    if not affinity_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if affinity_ready:
        cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)
    if not affinity_ready:
        utils.format_output("Affinity scenario", {"reason": "No eligible multi-backend Responses pool; no tests executed."}, status="SKIP", part="RESULT")


## ✅ Summary

This notebook validated **backend-pool session affinity** for the stateful OpenAI Responses API:
- Enabled `sessionAwareModel: true` for the target model and redeployed the LLM backend onboarding (`.bicepparam` in `validation/.runs/<scenario>/<run-id>/`).
- Provisioned an access contract scoped to the target model with response headers enabled.
- Ran 4 sessions × 3 requests using the OpenAI SDK over a per-session `httpx` cookie jar; the first request created a response id and the follow-ups reused it.
- Visualized the response id, backend, and affinity cookie per request, and asserted that each session stuck to one backend with a maintained `ai-gateway-affinity` cookie.

See [LLM Backend Onboarding — Session Affinity Configuration](../bicep/infra/llm-backend-onboarding/README.md#session-affinity-configuration) and [LLM Access Guide — Backend Pool Types](../guides/llm-access-guide.md#backend-pool-types) for the underlying configuration and client cookie-jar guidance.
